# PCam binary classifier — Kaggle T4 GPU
Training ResNet-18 on PatchCamelyon for tumour/normal patch classification.
Artifacts are saved to `/kaggle/working/` and can be downloaded after the run.

In [ ]:
# Install dependencies
import subprocess
subprocess.run([
    'pip', 'install', '-q',
    'torch', 'torchvision', 'scikit-learn', 'numpy', 'h5py'
], check=True)
print('Dependencies installed')

In [ ]:
import os
import time
import json
import logging
from dataclasses import dataclass
from pathlib import Path
from typing import Tuple

import h5py
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix
from PIL import Image

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger(__name__)

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Locate the PCam HDF5 files on Kaggle
DATA_DIR = Path('/kaggle/input/metastatic-tissue-classification-patchcamelyon')
OUTPUT_DIR = Path('/kaggle/working/checkpoints')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# List available files
for f in sorted(DATA_DIR.rglob('*.h5')):
    print(f)

In [ ]:
@dataclass(frozen=True)
class TrainingConfig:
    """
    Immutable training configuration — same design as train.py.
    Kaggle-specific paths are set here; everything else is identical
    to the Dardel version so the two scripts stay in sync.
    """
    output_dir: str  = str(OUTPUT_DIR)
    epochs: int      = 5
    batch_size: int  = 128
    learning_rate: float = 1e-4
    num_workers: int = 2
    log_every_n_steps: int = 200

    @property
    def device(self) -> torch.device:
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

cfg = TrainingConfig()
print(f'Device: {cfg.device}')
print(f'Config: {cfg}')

In [ ]:
class PCamHDF5Dataset(Dataset):
    """
    Reads PatchCamelyon directly from the HDF5 files provided in the
    Kaggle dataset. Each sample is a 96x96 RGB patch.

    The Kaggle dataset stores images in camelyonpatch_level_2_split_*_x.h5
    and labels in camelyonpatch_level_2_split_*_y.h5.

    We load lazily (one sample at a time) to avoid OOM on large splits.
    """

    def __init__(self, x_path: Path, y_path: Path, transform):
        self.x_path    = x_path
        self.y_path    = y_path
        self.transform = transform
        # Open once to get length, close immediately
        with h5py.File(x_path, 'r') as f:
            self.length = f['x'].shape[0]

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        with h5py.File(self.x_path, 'r') as fx, h5py.File(self.y_path, 'r') as fy:
            image = fx['x'][idx]          # (96, 96, 3) uint8
            label = int(fy['y'][idx][0][0][0])
        image = Image.fromarray(image)
        return self.transform(image), label


def get_transforms():
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    train_tf = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    val_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    return train_tf, val_tf


def get_loaders(cfg: TrainingConfig) -> Tuple[DataLoader, DataLoader]:
    train_tf, val_tf = get_transforms()
    train_ds = PCamHDF5Dataset(
        DATA_DIR / 'camelyonpatch_level_2_split_train_x.h5',
        DATA_DIR / 'camelyonpatch_level_2_split_train_y.h5',
        train_tf,
    )
    val_ds = PCamHDF5Dataset(
        DATA_DIR / 'camelyonpatch_level_2_split_valid_x.h5',
        DATA_DIR / 'camelyonpatch_level_2_split_valid_y.h5',
        val_tf,
    )
    train_loader = DataLoader(
        train_ds, batch_size=cfg.batch_size,
        shuffle=True, num_workers=cfg.num_workers, pin_memory=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg.batch_size,
        shuffle=False, num_workers=cfg.num_workers, pin_memory=True,
    )
    log.info(f'Train: {len(train_ds)} samples, Val: {len(val_ds)} samples')
    return train_loader, val_loader

In [ ]:
def build_model() -> nn.Module:
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model


def train_epoch(model, loader, criterion, optimizer, device, log_every):
    model.train()
    total_loss = 0.0
    for step, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        if step % log_every == 0:
            log.info(f'  step {step}/{len(loader)}  loss={loss.item():.4f}')
    return total_loss / len(loader)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_labels, all_probs, all_preds, latencies = 0.0, [], [], [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            t0     = time.perf_counter()
            logits = model(images)
            latencies.append((time.perf_counter() - t0) * 1000)
            total_loss += criterion(logits, labels).item()
            all_probs.extend(torch.softmax(logits, 1)[:, 1].cpu().numpy())
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    all_labels = np.array(all_labels)
    all_preds  = np.array(all_preds)
    all_probs  = np.array(all_probs)
    return {
        'loss':                 round(total_loss / len(loader), 4),
        'accuracy':             round(float((all_preds == all_labels).mean()), 4),
        'auc':                  round(float(roc_auc_score(all_labels, all_probs)), 4),
        'f1':                   round(float(f1_score(all_labels, all_preds)), 4),
        'confusion_matrix':     confusion_matrix(all_labels, all_preds).tolist(),
        'latency_ms_per_batch': round(float(np.mean(latencies)), 2),
    }

In [ ]:
# --- Main training loop ------------------------------------------------------

device        = cfg.device
model         = build_model().to(device)
criterion     = nn.CrossEntropyLoss()
optimizer     = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
train_loader, val_loader = get_loaders(cfg)

best_auc    = 0.0
all_metrics = []

for epoch in range(1, cfg.epochs + 1):
    log.info(f'Epoch {epoch}/{cfg.epochs}')
    train_loss  = train_epoch(model, train_loader, criterion, optimizer, device, cfg.log_every_n_steps)
    val_metrics = evaluate(model, val_loader, criterion, device)
    val_metrics.update({'epoch': epoch, 'train_loss': round(train_loss, 4)})
    all_metrics.append(val_metrics)

    log.info(
        f"  val_loss={val_metrics['loss']}  "
        f"acc={val_metrics['accuracy']}  "
        f"auc={val_metrics['auc']}  "
        f"f1={val_metrics['f1']}"
    )

    if val_metrics['auc'] > best_auc:
        best_auc = val_metrics['auc']
        torch.save(model.state_dict(), OUTPUT_DIR / 'best_model.pt')
        log.info(f'  New best AUC={best_auc:.4f} — checkpoint saved')

torch.save(model.state_dict(), OUTPUT_DIR / 'final_model.pt')

with open(OUTPUT_DIR / 'metrics.json', 'w') as f:
    json.dump(all_metrics, f, indent=2)

with open(OUTPUT_DIR / 'config.json', 'w') as f:
    json.dump(cfg.__dict__, f, indent=2)

print('Training complete!')
print(json.dumps(all_metrics[-1], indent=2))

In [ ]:
# List output artifacts
for f in sorted(OUTPUT_DIR.iterdir()):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f'{f.name:30s}  {size_mb:.1f} MB')